# Bronze Layer: Patient Data Ingestion

## Purpose
Ingest raw patient data from Unity Catalog Volume with:
* **Schema inference** from CSV headers
* **Metadata tracking** (source file, ingestion timestamp)
* **Quality validation** (duplicates, nulls)
* **Production-ready** defensive write pattern

## Data Flow
```
Volume CSV → DataFrame → Bronze Delta Table
/Volumes/.../patients.csv → healthcare.bronze.patient_raw
```

## Professional Exam Topics
* **Batch ingestion patterns** - Simple CSV → Delta
* **Unity Catalog integration** - Volume-based ingestion
* **Audit metadata** - Tracking data lineage
* **Quality checks** - Validation before write
* **Delta Lake basics** - Overwrite vs append modes

In [0]:
# Centralized configuration
from pyspark.sql import functions as F

# Source and target configuration
SOURCE_PATH = "/Volumes/healthcare/bronze/raw_files/patients.csv"
TARGET_TABLE = "healthcare.bronze.patient_raw"

print("="*60)
print("🏥 BRONZE LAYER: PATIENT INGESTION")
print("="*60)
print(f"\n📂 Source: {SOURCE_PATH}")
print(f"🎯 Target: {TARGET_TABLE}")
print(f"\n✅ Configuration loaded")

In [0]:
# Read CSV with schema inference and add metadata columns
df_patient = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load(SOURCE_PATH)
         .withColumn("source_file", F.col("_metadata.file_path"))
         .withColumn("ingestion_timestamp", F.current_timestamp())
)

print("\n📊 Schema:")
df_patient.printSchema()

# Single-pass quality validation
quality_metrics = df_patient.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("Id").alias("unique_ids"),
    F.sum(F.when(F.col("Id").isNull(), 1).otherwise(0)).alias("null_ids"),
    F.sum(F.when(F.col("BIRTHDATE").isNull(), 1).otherwise(0)).alias("null_birthdates"),
    F.sum(F.when(F.col("GENDER").isNull(), 1).otherwise(0)).alias("null_genders")
).collect()[0]

print("\n🔍 Quality Validation:")
print("="*60)
print(f"Total Rows: {quality_metrics['total_rows']}")
print(f"Unique Patient IDs: {quality_metrics['unique_ids']}")
print(f"Duplicate IDs: {quality_metrics['total_rows'] - quality_metrics['unique_ids']}")
print(f"NULL Patient IDs: {quality_metrics['null_ids']}")
print(f"NULL Birthdates: {quality_metrics['null_birthdates']}")
print(f"NULL Gender: {quality_metrics['null_genders']}")

# Validation status
has_duplicates = quality_metrics['total_rows'] != quality_metrics['unique_ids']
has_null_ids = quality_metrics['null_ids'] > 0

if has_duplicates:
    print("\n⚠️  WARNING: Duplicate patient IDs detected")
if has_null_ids:
    print("\n⚠️  WARNING: NULL patient IDs detected")
if not has_duplicates and not has_null_ids:
    print("\n✅ Data quality validation passed")

In [0]:
# Write to Bronze layer with metadata
print("\n💾 Writing to Bronze table...")

(
    df_patient.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TARGET_TABLE)
)

record_count = df_patient.count()
print(f"\n✅ Successfully wrote {record_count:,} records to {TARGET_TABLE}")
print(f"   - Includes source_file and ingestion_timestamp columns")
print(f"   - Mode: OVERWRITE (full refresh)")

In [0]:
# Verify Bronze table write
verify_df = spark.table(TARGET_TABLE)

print("\n✅ Bronze Table Verification:")
print("="*60)
print(f"Total records: {verify_df.count():,}")
print(f"Schema columns: {len(verify_df.columns)}")
print(f"\nMetadata columns present:")
print(f"  - source_file: {'✅' if 'source_file' in verify_df.columns else '❌'}")
print(f"  - ingestion_timestamp: {'✅' if 'ingestion_timestamp' in verify_df.columns else '❌'}")

print("\n📊 Sample data:")
display(verify_df.select("Id", "BIRTHDATE", "GENDER", "STATE", "source_file", "ingestion_timestamp").limit(5))